# Brain-state transitions vs. phenotype

Reads the tables written by `fmri-decomp transitions` and asks whether a
subject's transition structure tracks a phenotype column.

**Three sections, one per question.**

1. the per-subject transition table, as a matrix you can look at
2. a volcano: one dot per transition cell, across every state set
3. model selection: which state set predicts the phenotype best

**What this is.** camcan is the **discovery** cohort. The state set that wins
here is a candidate, and the CV score is a *ranking device* — not an effect
size. The confirmatory test is a different dataset, later. So the number to
quote from section 3 is "state set X ranked first", never "transitions explain
Y% of the variance".

Nothing here writes to `outputs/`.


In [ ]:
# ---- parameters -------------------------------------------------------------
import os
from pathlib import Path

ROOT = Path(os.environ.get("FMRIDECOMP_OUTPUTS",
                           "/project/6008063/tamires/DecomposingfMRI/outputs"))

# Phenotype: one row per subject. `PHENO_ID` is joined to the `sub=` partition
# key, so it must hold the bare id as a STRING -- CC110019, never parsed as a
# number.
# Cam-CAN's own release table: TSV, not CSV, so PHENO_SEP matters.
PHENO_PATH = Path("/project/6008063/tamires/cohorts/camcan/dataman/useraccess/"
                  "opendata/paule_toussaint_camcan01870/approved_data.tsv")
PHENO_SEP = "\t"
PHENO_ID = "CCID"
PHENO_COHORT = "camcan"

TARGETS = ["additional_HADS_anx_category", "additional_HADS_dep_category"]
TARGET = TARGETS[0]       # switch to TARGETS[1] and re-run the notebook

# Ordered, low to high. Position in this list IS the code: Normal=0 ... Severe=3.
# A value not in the list becomes NaN and is reported -- a fifth category, or a
# stray spelling, must be visible rather than silently dropped.
ORDINAL_LEVELS = ["Normal", "Mild", "Moderate", "Severe"]

# "ridge"    -> codes as continuous, scored by out-of-fold Spearman. Uses the
#               whole ordering, rank-based, and heavily regularised -- the most
#               reliable RANKER of state sets, which is this section's job.
# "boost"    -> gradient boosting (lightgbm if installed, else sklearn's
#               HistGradientBoosting, which is the same algorithm). Catches
#               interactions a linear model cannot. Note it is higher-variance
#               at n~600 with 64-729 features, so the state-set ranking wobbles
#               more between fold seeds -- watch the right-hand panel.
# "logistic" -> Normal vs anything above it, scored by AUC. Better powered when
#               the upper levels are thin, but throws the ordering away.
MODEL = "ridge"

# Nuisance. age is not optional: camcan spans 18-88, and both brain dynamics and
# HADS move with it, so without it the volcano reports age effects wearing a
# HADS label. n_transitions varies because censoring removed windows unevenly.
COVARIATES = ["age", "sex", "n_transitions"]

P_COLUMN = "p_joint"      # or "p_cond" -- see section 1
N_PERM = 1000
SEEDS = [0, 1, 2, 3, 4]   # the state-set ranking is repeated over these, to see
                          # whether the winner is stable or noise

In [ ]:
import itertools, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

TRANS = ROOT / "transitions"
if not TRANS.is_dir():
    raise SystemExit(f"{TRANS} does not exist -- run `fmri-decomp transitions` first")

# Every state set that actually has tables, discovered from the tree.
sets = pd.DataFrame(
    [{"atlas": p.parts[-5].split("=")[1], "window_s": p.parts[-4].split("=")[1],
      "states": p.parts[-3].split("=")[1], "cohort": p.parts[-2].split("=")[1],
      "path": p.parent}
     for p in TRANS.rglob("subjects.parquet")])
sets["K"] = sets["states"].str.rsplit("_", n=1).str[1].astype(int)
sets["window_s_num"] = sets["window_s"].astype(float)

print(f"{len(sets)} (state set x cohort) table(s) on disk\n")
print(sets.groupby(["atlas", "window_s_num", "K"]).size()
          .rename("cohorts").reset_index().to_string(index=False))

## 1. The transition table

One row per subject, one table. Columns in three blocks:

```
cohort task sub | 0->0 0->1 ... (every K*K cell) | switch_rate ... | provenance
```

Cell values are `p_joint` — that subject's count for `i->j` divided by all of
their transitions, so the cells sum to 1 across the row. Unobserved transitions
are exact zeros, not missing.

Nothing is lost by storing one normalisation: `count = p_joint * n_transitions`
exactly, and `p_cond` (the Markov probability, each row summing to 1) is the
row-wise renormalisation below.

In [ ]:
def load(atlas, window_s, states, cohort):
    """The one table for a (state set, cohort). One row per subject."""
    return pd.read_parquet(TRANS / f"atlas={atlas}" / f"window_s={window_s}"
                           / f"states={states}" / f"cohort={cohort}"
                           / "subjects.parquet")

def cell_cols(d):
    """The transition columns, in (from, to) order."""
    return sorted((c for c in d.columns if "->" in c),
                  key=lambda c: tuple(int(x) for x in c.split("->")))

def as_p_cond(d):
    """Cells renormalised per from-state: the Markov transition probability."""
    cols, K = cell_cols(d), int(d["n_states"].iloc[0])
    M = d[cols].to_numpy(float).reshape(len(d), K, K)
    M = M * d["n_transitions"].to_numpy(float)[:, None, None]      # back to counts
    rowsum = M.sum(axis=2, keepdims=True)
    out = np.divide(M, rowsum, out=np.zeros_like(M), where=rowsum > 0)
    return pd.DataFrame(out.reshape(len(d), K * K), columns=cols, index=d.index)

demo = sets[sets["cohort"] == PHENO_COHORT].sort_values(["K", "window_s_num"]).iloc[0]
print(f"showing: atlas={demo.atlas}  window_s={demo.window_s}  "
      f"{demo.states}  cohort={demo.cohort}")
d0 = load(demo.atlas, demo.window_s, demo.states, demo.cohort)
cols0 = cell_cols(d0)
print(f"{d0.shape[0]} subject(s) x {d0.shape[1]} column(s)  ({len(cols0)} cells)")
print(f"zeros among the cells: {100 * (d0[cols0] == 0).to_numpy().mean():.1f}%")
d0[["sub"] + cols0[:8] + ["n_transitions", "switch_rate", "mean_dwell_s"]].head(8).round(3)

## 2. Volcano — every transition cell, every state set

One dot per `(state set, transition cell)`. The test runs **across subjects**:
each subject contributes one value of `p` for that cell, and the question is
whether it tracks the phenotype. So **n = subjects, and subjects are
independent** — the overlapping windows live inside each subject's number and
cost precision, not validity.

Both axes come from a **partial** correlation: phenotype and cell are each
residualised on the covariates first, so what is plotted is the association
that age, sex and transition count do not already explain.

Two thresholds are drawn. **BH-FDR** at 5% on the analytic p-values, and a
**permutation FWER** line from the null distribution of the largest |r| over
every cell — the second is the stricter one and needs no independence
assumption across cells.

Cells that are zero for every subject have no variance, cannot be tested, and
are dropped before either correction, so they do not inflate the family.


In [ ]:
def design(cov: pd.DataFrame) -> np.ndarray:
    """Covariates -> a numeric design matrix with an intercept."""
    X = pd.get_dummies(cov, drop_first=True, dummy_na=False).astype(float)
    X = X.fillna(X.mean())
    return np.column_stack([np.ones(len(X)), X.to_numpy()])

def residualise(Y: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Columns of Y with the span of X projected out."""
    beta, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return Y - X @ beta

def _z(A):
    """Standardise columns, leaving zero-variance ones as exact zeros.

    `out=` is not optional: np.divide with `where=` and no `out` leaves the
    masked entries UNINITIALISED, so a flat column would carry whatever was in
    memory. They are dropped downstream by `keep`, but a correct value beats
    relying on that.
    """
    A = A - A.mean(0)
    sd = A.std(0)
    return np.divide(A, sd, out=np.zeros_like(A), where=sd > 0), sd

def partial_corr_scan(cells_wide: pd.DataFrame, y: np.ndarray, X: np.ndarray,
                      n_perm: int = N_PERM, seed: int = 0):
    """Partial correlation of every column against y, plus a permutation null.

    Vectorised on purpose: residualise once, standardise once, then every
    correlation is one matrix product. The permutation null shuffles the
    residualised phenotype, which is the exchangeable quantity once the
    covariates are out.
    """
    A = residualise(cells_wide.to_numpy(float), X)
    yr = residualise(y.reshape(-1, 1), X).ravel()
    Az, sd = _z(A)
    keep = sd > 0                                  # no variance -> untestable
    Az = Az[:, keep]
    yz = (yr - yr.mean()) / yr.std()
    n = len(yz)
    r = (Az.T @ yz) / n

    df = n - X.shape[1] - 1
    with np.errstate(divide="ignore", invalid="ignore"):
        t = r * np.sqrt(df / (1 - r ** 2))
    p = 2 * stats.t.sf(np.abs(t), df)

    rng = np.random.default_rng(seed)
    max_null = np.empty(n_perm)
    for b in range(n_perm):
        max_null[b] = np.abs((Az.T @ rng.permutation(yz)) / n).max()

    out = pd.DataFrame({"transition": cells_wide.columns[keep], "r": r, "p": p})
    out["p_fdr"] = stats.false_discovery_control(out["p"], method="bh")
    return out, float(np.quantile(max_null, 0.95)), int((~keep).sum())

In [ ]:
# Phenotype
if not PHENO_PATH.exists():
    raise SystemExit(
        f"{PHENO_PATH} does not exist.\n"
        f"Build it with tools/make_phenotype.py, or point PHENO_PATH elsewhere.")
pheno = pd.read_csv(PHENO_PATH, sep=PHENO_SEP, dtype={PHENO_ID: str})
for c in (PHENO_ID, TARGET):
    if c not in pheno.columns:
        raise SystemExit(f"{c!r} is not in {PHENO_PATH.name}: "
                         f"{list(pheno.columns)[:20]}")
pheno = pheno.rename(columns={PHENO_ID: "sub"})
print(f"phenotype: {len(pheno):,} row(s) from {PHENO_PATH.name}")

# Ordered categories -> integer codes. Not pd.factorize, which numbers by order
# of appearance; the clinical ordering has to be the one we assert.
raw = pheno[TARGET].astype("string").str.strip()
codes = raw.map({lvl: k for k, lvl in enumerate(ORDINAL_LEVELS)})
unknown = sorted(set(raw.dropna()) - set(ORDINAL_LEVELS))
if unknown:
    print(f"  WARNING: value(s) not in ORDINAL_LEVELS, now NaN: {unknown}")
pheno["y"] = codes.astype("Float64")

print(f"\n  {TARGET}")
dist = raw.value_counts(dropna=False).reindex(ORDINAL_LEVELS + [np.nan])
for lvl, n in dist.items():
    code = ORDINAL_LEVELS.index(lvl) if lvl in ORDINAL_LEVELS else "-"
    n = 0 if pd.isna(n) else int(n)
    print(f"    {str(lvl):<10} code {str(code):<3} {n:>6,}")
pheno = pheno.dropna(subset=["y"])
pheno["y"] = pheno["y"].astype(int)
print(f"  -> {len(pheno):,} row(s) with a usable label")

missing = [c for c in COVARIATES if c != "n_transitions" and c not in pheno.columns]
if missing:
    raise SystemExit(f"covariate(s) not in {PHENO_PATH.name}: {missing}")

# `sub` must be the same identifier on both sides, so normalise and then CHECK.
pheno["sub"] = pheno["sub"].astype(str).str.strip().str.upper()
for c in COVARIATES:
    if c in pheno.columns and c != "sex":
        pheno[c] = pd.to_numeric(pheno[c], errors="coerce")


def feature_frame(atlas, window_s, states, cohort=PHENO_COHORT):
    """One transition table joined to phenotype on `sub`. Returns (frame, cells)."""
    t = load(atlas, window_s, states, cohort)
    if P_COLUMN == "p_cond":
        t[cell_cols(t)] = as_p_cond(t)
    t["sub"] = t["sub"].astype(str).str.strip().str.upper()
    d = t.merge(pheno, on="sub", how="inner", suffixes=("", "_pheno"))
    if d.empty:
        raise SystemExit(
            "the phenotype join matched NOTHING -- the identifiers differ.\n"
            f"  transitions sub e.g. {t['sub'].iloc[:3].tolist()}\n"
            f"  phenotype   sub e.g. {pheno['sub'].iloc[:3].tolist()}")
    return d, cell_cols(t)


grid = (sets[sets["cohort"] == PHENO_COHORT]
        .sort_values(["atlas", "K", "window_s_num"]))
print(f"\nscanning {len(grid)} state set(s) against {TARGET}\n")

scans, fwer = [], {}
for t in grid.itertuples():
    d, cols = feature_frame(t.atlas, t.window_s, t.states)
    if len(d) < 30:
        print(f"  skip {t.atlas}/{t.window_s}s/K={t.K}: only {len(d)} subject(s)")
        continue
    X = design(d[COVARIATES])
    # The target is a 4-level ordinal, so rank it: this makes the scan a PARTIAL
    # SPEARMAN, which does not assume the gap Normal->Mild equals Moderate->Severe.
    y = stats.rankdata(d["y"].to_numpy(float))
    res, thr, n_flat = partial_corr_scan(d[cols], y, X)
    scans.append(res.assign(atlas=t.atlas, window_s=t.window_s_num, K=t.K,
                            n=len(d)))
    fwer[(t.atlas, t.window_s_num, t.K)] = thr
    print(f"  {t.atlas:>14} {t.window_s:>4}s K={t.K:<3} n={len(d):<4} "
          f"{len(res):>5} testable cell(s) ({n_flat} flat)  "
          f"perm |r| 95% = {thr:.3f}")

volc = pd.concat(scans, ignore_index=True)
volc["p_fdr_all"] = stats.false_discovery_control(volc["p"], method="bh")
print(f"\nfamily: {len(volc):,} testable cell(s) across every state set")
print(f"surviving BH-FDR 5% over the whole family: {(volc['p_fdr_all'] < .05).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
colour = {8: "tab:blue", 27: "tab:orange"}
for K, g in volc.groupby("K"):
    ax.scatter(g["r"], -np.log10(g["p"]), s=6, alpha=.35,
               c=colour.get(K, "grey"), label=f"K={K}  ({len(g):,} cells)")

sig = volc[volc["p_fdr_all"] < .05]
if len(sig):
    ax.scatter(sig["r"], -np.log10(sig["p"]), s=26, facecolors="none",
               edgecolors="k", linewidths=.8, label=f"BH-FDR 5% ({len(sig)})")
    for t in sig.nsmallest(8, "p").itertuples():
        ax.annotate(f"{t.transition}\n{t.atlas} {t.window_s:g}s K={t.K}",
                    (t.r, -np.log10(t.p)), fontsize=6,
                    xytext=(4, 2), textcoords="offset points")

if len(volc):
    worst = volc.loc[volc["p_fdr_all"] < .05, "p"].max() if len(sig) else None
    if worst:
        ax.axhline(-np.log10(worst), ls="--", lw=.8, c="k",
                   label="BH-FDR 5% threshold")
ax.axvline(0, lw=.6, c="grey")
ax.set_xlabel(f"partial r  (adjusted for {', '.join(COVARIATES)})")
ax.set_ylabel(r"$-\log_{10}$ p")
ax.set_title(f"{TARGET} vs transition cells — every state set\n"
             f"family = {len(volc):,} tests, n = subjects (independent)", fontsize=10)
ax.legend(fontsize=7, loc="upper left")
fig.tight_layout()

print(volc.nsmallest(12, "p")[["atlas","window_s","K","transition","r","p","p_fdr_all"]]
          .to_string(index=False))

## 3. Model selection — which state set predicts best

For each state set: the transition cells plus covariates go into a ridge, scored
out-of-fold. Repeated over `SEEDS`, because a ranking that reshuffles with the
fold seed has not selected anything.

Ridge rather than OLS because the cells are many, correlated and compositional
(each subject's row sums to 1). Covariates are always in the model, so a state
set has to beat age and sex rather than rediscover them — `covariates only` is
the baseline line on the plot.

The score is **out-of-fold Spearman** between predicted and observed: rank-based,
so it is not dragged by HADS's floor, and it is a ranking statistic, which is
all this is for.


In [ ]:
from sklearn.linear_model import RidgeCV, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ALPHAS = np.logspace(-2, 4, 25)


def _boost():
    """lightgbm if it is installed, else sklearn's HistGradientBoosting.

    Same algorithm -- sklearn's implementation is histogram-based boosting
    modelled on LightGBM -- so the fallback is not a downgrade in kind. The
    stage45 container does not ship lightgbm; `containers/stage45.def` has it
    for the next rebuild.

    The hyperparameters are deliberately small. Defaults are tuned for tens of
    thousands of rows; here it is ~600 subjects against 64-729 correlated,
    compositional features, where a default-sized forest memorises the training
    fold and the state-set ranking turns to noise.
    """
    kw = dict(learning_rate=0.03, random_state=0)
    try:
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=400, num_leaves=7, min_child_samples=30,
                             colsample_bytree=0.5, subsample=0.8, subsample_freq=1,
                             reg_lambda=1.0, verbose=-1, **kw), "lightgbm"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(max_iter=400, max_leaf_nodes=7,
                                             min_samples_leaf=30,
                                             l2_regularization=1.0,
                                             early_stopping=False, **kw), \
               "sklearn HistGradientBoosting (lightgbm not installed)"


def estimator():
    if MODEL == "ridge":
        return make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS)), "ridge"
    if MODEL == "boost":
        m, name = _boost()
        return m, name
    if MODEL == "logistic":
        return make_pipeline(StandardScaler(),
                             LogisticRegression(max_iter=2000, C=0.1,
                                                class_weight="balanced")), "logistic"
    raise SystemExit(f"MODEL must be ridge, boost or logistic, got {MODEL!r}")


EST, EST_NAME = estimator()
print(f"MODEL = {MODEL!r} -> {EST_NAME}")
print("metric = out-of-fold "
      + ("AUC (Normal vs above)" if MODEL == "logistic" else "Spearman"))


def oof_score(Xm, y, seed):
    """One number per (state set, seed). Never accuracy: with the upper levels
    this thin, predicting Normal for everyone scores well and means nothing."""
    from sklearn.base import clone
    if MODEL == "logistic":
        yb = (y > 0).astype(int)
        pred = np.empty(len(yb), float)
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xm, yb):
            m = clone(EST).fit(Xm[tr], yb[tr])
            pred[te] = m.predict_proba(Xm[te])[:, 1]
        return roc_auc_score(yb, pred)
    pred = np.empty(len(y), float)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(Xm):
        pred[te] = clone(EST).fit(Xm[tr], y[tr]).predict(Xm[te])
    return stats.spearmanr(pred, y).statistic


rows = []
for t in grid.itertuples():
    d, cols = feature_frame(t.atlas, t.window_s, t.states)
    if len(d) < 50:
        continue
    y = d["y"].to_numpy(float)
    C = design(d[COVARIATES])[:, 1:]                 # drop the intercept
    F = np.column_stack([C, d[cols].to_numpy(float)])
    for seed in SEEDS:
        rows.append({"atlas": t.atlas, "window_s": t.window_s_num, "K": t.K,
                     "n": len(d), "n_cells": len(cols), "seed": seed,
                     "score": oof_score(F, y, seed)})
    for seed in SEEDS:                               # the baseline to beat
        rows.append({"atlas": t.atlas, "window_s": t.window_s_num, "K": 0,
                     "n": len(d), "n_cells": 0, "seed": seed,
                     "score": oof_score(C, y, seed)})

perf = pd.DataFrame(rows)
agg = (perf.groupby(["atlas", "window_s", "K"])["score"]
            .agg(["mean", "std", "min", "max"]).round(3).reset_index()
            .sort_values("mean", ascending=False))
print(f"\ntarget = {TARGET}   (K=0 rows are covariates only)\n")
print(agg.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for (atlas, K), g in perf[perf["K"] > 0].groupby(["atlas", "K"]):
    m = g.groupby("window_s")["score"].agg(["mean", "std"])
    ax.errorbar(m.index, m["mean"], yerr=m["std"], marker="o", capsize=3,
                label=f"{atlas} K={K}")
base = perf[perf["K"] == 0].groupby("window_s")["score"].mean()
ax.plot(base.index, base.values, "k--", lw=1.2, label="covariates only")
ax.axhline(0, lw=.6, c="grey")
ax.set_xscale("log"); ax.set_xticks(sorted(perf["window_s"].unique()))
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlabel("window_s"); ax.set_ylabel(f"out-of-fold "
                  f"{'AUC' if MODEL == 'logistic' else 'Spearman'}")
ax.set_title(f"{TARGET}: performance by aperture\n(bars = SD over "
             f"{len(SEEDS)} fold seeds)", fontsize=10)
ax.legend(fontsize=7)

# Is the winner stable, or does it change with the fold seed?
ax = axes[1]
rank = (perf[perf["K"] > 0]
        .assign(label=lambda d: d["atlas"] + " " + d["window_s"].astype(str)
                + "s K" + d["K"].astype(str))
        .pivot_table(index="label", columns="seed", values="score")
        .rank(ascending=False))
order = rank.mean(axis=1).sort_values().index
ax.imshow(rank.loc[order], aspect="auto", cmap="viridis_r")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=7)
ax.set_xticks(range(len(SEEDS))); ax.set_xticklabels(SEEDS)
ax.set_xlabel("fold seed"); ax.set_title("rank by seed (1 = best)\n"
                                         "rows that change colour are not a "
                                         "stable choice", fontsize=10)
for i, lab in enumerate(order):
    for j, s in enumerate(SEEDS):
        ax.text(j, i, int(rank.loc[lab, s]), ha="center", va="center",
                fontsize=6, color="w")
fig.tight_layout()

## What this licenses, and what it does not

* **The winner is a candidate.** Ranked first on camcan, which is the discovery
  cohort. Confirmation is a different dataset.
* **Do not quote the CV score as an effect size.** Choosing the best of ~30
  state sets biases the winner's score upward even when nothing is real. The
  score ordered the candidates; that is its whole job.
* **Check the right-hand panel before believing the left.** If a state set's
  rank moves across fold seeds, the comparison did not separate it from its
  neighbours and "best" is noise.
* **It has to beat `covariates only`.** A state set below the dashed line is
  doing worse than age and sex alone.
* **The overlap is in the table, not in your memory.**
  `n_transitions_independent` beside `n_transitions` is what a non-overlapping
  grid would have given — about a fifth. Transitions are correlated within a
  subject, which costs power and cannot manufacture a false positive, because
  every test here has the subject as its unit.
* **The cells are compositional, so hits come in sprays.** Each subject's row
  of `p_joint` sums to 1, so inflating one cell necessarily deflates the others.
  Simulating a single planted effect on `0->0` gave 3 surviving cells when the
  effect was weak and **all 64** when it was strong — one cause, sixty-four
  dots. So "how many cells survived" is not "how many findings"; read the
  volcano for *which* cell is strongest, and treat its neighbours as the same
  result seen sideways. (Null simulation gave 0 of 64, so the test itself is
  calibrated.)
* **`additional_HADS_depression` is thin.** Only ~9 camcan subjects reach the
  conventional cut of 11, so treat it as continuous and floor-limited, and do
  not dichotomise it.
